In [1]:
# ════════════════════════════════════════════════════════════════
# NOTEBOOK : augmentation_donnees.ipynb
# Objectif : rééquilibrer le dataset par augmentation audio
# ════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import librosa
import soundfile as sf
import json, os
from collections import Counter
from pathlib import Path

# ── Chemins ───────────────────────────────────────────────────
import librosa
import soundfile as sf
from pathlib import Path
import os

RAW_DIR = Path("C:/Users/asalou/S7/stages7/project/raw_audio")
WAV_DIR = Path("C:/Users/asalou/S7/stages7/project/wav_audio")
OUTPUT_DIR = Path("C:/Users/asalou/S7/stages7/project/augmentation")
AUG_DIR    = f"{OUTPUT_DIR}/augmented_segments"
os.makedirs(AUG_DIR, exist_ok=True)

TARGET_SR    = 16000
SEGMENT_MS   = 500
SEG_LEN      = int(SEGMENT_MS * TARGET_SR / 1000)  # 8000 samples


dataset = {}
for file in WAV_DIR.glob("*.wav"):
    y, sr = librosa.load(file, sr=16000)

    dataset[file.stem] = {
        'data': y,
        'filtered': y.copy(),   
        'sr': sr,
        'duration': len(y)/sr,
        'energy': None,
        'times': None,
        'threshold': None,
        'segments': []
    }
print("Dataset chargé :", len(dataset))

# ── Charger le dataset labélisé ───────────────────────────────
with open(f'{OUTPUT_DIR}/labels_segments.json') as f:
    labels_check = json.load(f)

# ── Objectif : ~750 segments par classe ───────────────────────
TARGET_PER_CLASS = 750

Dataset chargé : 21


In [2]:
# ════════════════════════════════════════════════════════════════
# FONCTIONS D'AUGMENTATION
# ════════════════════════════════════════════════════════════════

def aug_bruit(seg, sr, sigma=0.005):
    """Ajout de bruit gaussien."""
    return seg + np.random.normal(0, sigma, len(seg))

def aug_gain(seg, sr, low=0.7, high=1.3):
    """Variation de gain aléatoire."""
    return seg * np.random.uniform(low, high)

def aug_shift(seg, sr, max_ms=50):
    """Décalage temporel aléatoire."""
    shift = int(np.random.uniform(-max_ms, max_ms) * sr / 1000)
    return np.roll(seg, shift)

def aug_stretch(seg, sr, low=0.9, high=1.1):
    """Time stretching puis recadrage à la longueur originale."""
    rate      = np.random.uniform(low, high)
    stretched = librosa.effects.time_stretch(seg.astype(float), rate=rate)
    # Recadrer à SEG_LEN
    if len(stretched) >= SEG_LEN:
        return stretched[:SEG_LEN]
    else:
        return np.pad(stretched, (0, SEG_LEN - len(stretched)))

def aug_pitch(seg, sr, steps=(-2, -1, 1, 2)):
    """Pitch shifting d'un nombre de demi-tons aléatoire."""
    n = np.random.choice(steps)
    return librosa.effects.pitch_shift(seg.astype(float), sr=sr, n_steps=n)

def augmenter_segment(seg, sr, methode=None):
    """
    Applique une augmentation aléatoire ou spécifique.
    Combine toujours bruit + une autre technique.
    """
    methodes = [aug_gain, aug_shift, aug_stretch, aug_pitch]
    if methode is None:
        methode = np.random.choice(methodes)
    seg_aug = methode(seg, sr)
    seg_aug = aug_bruit(seg_aug, sr, sigma=np.random.uniform(0.001, 0.008))
    # Renormaliser
    mx = np.max(np.abs(seg_aug))
    if mx > 0:
        seg_aug = seg_aug / mx * np.max(np.abs(seg))
    return seg_aug.astype(np.float32)

In [3]:
# ════════════════════════════════════════════════════════════════
# ════════════════════════════════════════════════════════════════

# ── Reconstruire les segments depuis df_features ──────────────
# df_features contient les features MAIS pas le signal brut
# On reconstruit depuis dataset + labels_check directement

segments_par_classe = {
    'miction_active': [],
    'bruit_ambiant':  [],
    'bruit_chasse':   [],
    'silence':        [],
}

for nom_fichier, labels in labels_check.items():
    if nom_fichier not in dataset:
        continue
    filt = dataset[nom_fichier]['filtered']
    for i, lbl in enumerate(labels):
        if lbl not in segments_par_classe:
            continue
        start = i * SEG_LEN
        end   = start + SEG_LEN
        if end > len(filt):
            break
        segments_par_classe[lbl].append(filt[start:end].copy())

print("Segments disponibles par classe :")
for lbl, segs in segments_par_classe.items():
    print(f"  {lbl:20s} : {len(segs):4d} segments")

# ── Augmentation vers TARGET_PER_CLASS ────────────────────────
TARGET_PER_CLASS = 750
np.random.seed(42)

segments_liste = []
labels_liste   = []

print(f"\nAugmentation → {TARGET_PER_CLASS} segments par classe")
print("="*55)

for lbl, segs_orig in segments_par_classe.items():
    n_orig   = len(segs_orig)
    n_needed = max(0, TARGET_PER_CLASS - n_orig)
    tous     = list(segs_orig)

    for _ in range(n_needed):
        seg_src = segs_orig[np.random.randint(len(segs_orig))]
        seg_aug = augmenter_segment(seg_src, TARGET_SR)
        tous.append(seg_aug)

    segments_liste.extend(tous)
    labels_liste.extend([lbl] * len(tous))
    print(f"  {lbl:20s} : {n_orig} originaux + {n_needed} augmentés = {len(tous)}")

print("="*55)

# ── Sauvegarder ───────────────────────────────────────────────
X_aug = np.array(segments_liste, dtype=np.float32)  # (N, 8000) signaux bruts
y_aug = np.array(labels_liste)

np.save(f'{OUTPUT_DIR}/X_augmented.npy', X_aug)
np.save(f'{OUTPUT_DIR}/y_augmented.npy', y_aug)

print(f"\n X_augmented.npy : {X_aug.shape}")
print(f"y_augmented.npy : {y_aug.shape}")
print(f"\nDistribution finale :")
print(pd.Series(y_aug).value_counts().to_string())

Segments disponibles par classe :
  miction_active       :  683 segments
  bruit_ambiant        :  463 segments
  bruit_chasse         :  222 segments
  silence              :  156 segments

Augmentation → 750 segments par classe
  miction_active       : 683 originaux + 67 augmentés = 750
  bruit_ambiant        : 463 originaux + 287 augmentés = 750
  bruit_chasse         : 222 originaux + 528 augmentés = 750
  silence              : 156 originaux + 594 augmentés = 750

 X_augmented.npy : (3000, 8000)
y_augmented.npy : (3000,)

Distribution finale :
miction_active    750
bruit_ambiant     750
bruit_chasse      750
silence           750
